In [1]:
%reload_ext autoreload
%autoreload 2

# get the dataset first

In [2]:
import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [3]:

import pandas as pd
import pke
from tqdm import tqdm

data_df = pd.read_parquet("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count.parquet").head(20)



def add_keyword_column(text, extractor):
    extractor.load_document(input=text, language="en")
    extractor.candidate_selection(n=1)
    extractor.candidate_weighting()
    keyphrases = extractor.get_n_best(n=len(extractor.weights))
    return keyphrases

extractor = pke.unsupervised.YAKE()

# Wrap the apply function with tqdm to display a progress bar and add a description
tqdm.pandas(desc="Extracting Keywords")
data_df['keywords'] = data_df['text'].progress_apply(lambda x: add_keyword_column(x, extractor))


Extracting Keywords: 100%|██████████| 20/20 [00:14<00:00,  1.41it/s]


In [5]:
data_df.head()

,url_1,first_field,second_field,third_field,fourth_field,old_text,normalized_url,collection__config_folder,url_2,generated_title,scraped_title,division_display,document_type_display,line,word_counts,text,keywords
0,https://www.earthsciweek.org/,Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week,TOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET INVOLVE...,http://earthsciweek.org,earth_science_week,https://www.earthsciweek.org/,,Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 3, 0, 7, ...","Earth Science Week\nCelebrate ""Earth Science E...","[(science, 0.025427747295594157), (earth, 0.02..."
1,https://www.earthsciweek.org/contests,Earth Science Week Contests,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week Contests,TOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET INVOLVE...,http://earthsciweek.org/contests,earth_science_week,https://www.earthsciweek.org/contests,,Earth Science Week Contests,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 76,...",Earth Science Week Contests\nBe part of Earth ...,"[(earth, 0.01769242434976565), (science, 0.026..."
2,https://www.earthsciweek.org/get-involved,Get Involved with Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Get Involved with Earth Science Week,TOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET INVOLVE...,http://earthsciweek.org/get-involved,earth_science_week,https://www.earthsciweek.org/get-involved,,Get Involved with Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 6, 0, 49,...",Get Involved with Earth Science Week\nWhether ...,"[(earth, 0.015650979329975796), (science, 0.01..."
3,https://www.earthsciweek.org/resources,Earth Science Week Resources,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week Resources,TOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET INVOLVE...,http://earthsciweek.org/resources,earth_science_week,https://www.earthsciweek.org/resources,,Earth Science Week Resources,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 45,...",Earth Science Week Resources\nThe Earth Scienc...,"[(earth, 0.008616909926985408), (science, 0.00..."
4,https://www.earthsciweek.org/support,Support Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Support Earth Science Week,TOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET INVOLVE...,http://earthsciweek.org/support,earth_science_week,https://www.earthsciweek.org/support,,Support Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 68,...",Support Earth Science Week\nThe annual celebra...,"[(earth, 0.01378861429370517), (science, 0.015..."


In [ ]:
import pandas as pd
import pke
from tqdm import tqdm
from multiprocessing import Pool

# Read data
data_df = pd.read_parquet("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count.parquet")

# Function to extract keywords
def add_keyword_column(text):
    extractor = pke.unsupervised.YAKE()
    extractor.load_document(input=text, language="en")
    extractor.candidate_selection(n=1)
    extractor.candidate_weighting()
    keyphrases = extractor.get_n_best(n=len(extractor.weights))
    return keyphrases

# Function to wrap apply using multiprocessing with tqdm progress bar
def apply_multiprocessing_with_progress(df, func, n_processes):
    with Pool(n_processes) as pool:
        # Wrap the pool.map call with tqdm to display a progress bar
        result = list(tqdm(pool.imap(func, df['text']), total=len(df), desc="Extracting Keywords"))
    return result

# Using multiprocessing to extract keywords
n_processes = 64  # Set the number of processes you want to use
data_df['yake'] = apply_multiprocessing_with_progress(data_df, add_keyword_column, n_processes)


Extracting Keywords:  50%|█████     | 48887/97443 [25:40<15:08, 53.47it/s]   

In [ ]:
data_df.to_parquet("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count_yake.parquet")

In [4]:
ddf.head()

,url_1,first_field,second_field,third_field,fourth_field,old_text,normalized_url,collection__config_folder,url_2,generated_title,scraped_title,division_display,document_type_display,line,word_counts,text,keywords
0,https://www.earthsciweek.org/,Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week,TOOLKIT CONTESTS WEBINARS GET INVOLVED RES...,http://earthsciweek.org,earth_science_week,https://www.earthsciweek.org/,,Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 3, 0, 7, ...","Earth Science Week Celebrate ""Earth Science Ev...","[(science, 0.025427747295594157), (earth, 0.02..."
1,https://www.earthsciweek.org/contests,Earth Science Week Contests,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week Contests,TOOLKIT CONTESTS WEBINARS GET INVOLVED RES...,http://earthsciweek.org/contests,earth_science_week,https://www.earthsciweek.org/contests,,Earth Science Week Contests,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 76,...",Earth Science Week Contests Be part of Earth S...,"[(earth, 0.01769242434976565), (science, 0.026..."
2,https://www.earthsciweek.org/get-involved,Get Involved with Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Get Involved with Earth Science Week,TOOLKIT CONTESTS WEBINARS GET INVOLVED RES...,http://earthsciweek.org/get-involved,earth_science_week,https://www.earthsciweek.org/get-involved,,Get Involved with Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 6, 0, 49,...",Get Involved with Earth Science Week Whether y...,"[(earth, 0.015650979329975796), (science, 0.01..."
3,https://www.earthsciweek.org/resources,Earth Science Week Resources,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Earth Science Week Resources,TOOLKIT CONTESTS WEBINARS GET INVOLVED RES...,http://earthsciweek.org/resources,earth_science_week,https://www.earthsciweek.org/resources,,Earth Science Week Resources,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 45,...",Earth Science Week Resources The Earth Science...,"[(earth, 0.008616909926985408), (science, 0.00..."
4,https://www.earthsciweek.org/support,Support Earth Science Week,/scrapers/earth_science_week/,/Earth Science/Earth Science Week/,Support Earth Science Week,TOOLKIT CONTESTS WEBINARS GET INVOLVED RES...,http://earthsciweek.org/support,earth_science_week,https://www.earthsciweek.org/support,,Support Earth Science Week,,,"[TOOLKIT, , CONTESTS, , WEBINARS, , GET INVOLV...","[1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 68,...",Support Earth Science Week The annual celebrat...,"[(earth, 0.01378861429370517), (science, 0.015..."


# testing script

In [3]:
import os
import json
import pandas as pd
import pke
from tqdm import tqdm
from multiprocessing import Pool
import functools

def add_yake_keyword_column(text):
    extractor = pke.unsupervised.YAKE()  # Initialize inside worker to avoid pickling issues
    extractor.load_document(input=text, language="en")
    extractor.candidate_selection(n=1)
    extractor.candidate_weighting()
    keyphrases = extractor.get_n_best(n=len(extractor.weights))
    keyphrases = [(str(keyphrase).strip().upper(), float(score)) for keyphrase, score in keyphrases]
    return keyphrases

# Function to apply multiprocessing with progress bar
def apply_multiprocessing_with_progress(df, func, n_processes, text_column):
    with Pool(n_processes) as pool:
        result = list(tqdm(pool.imap(func, df[text_column]), total=len(df), desc="Extracting Keywords"))
    return result


def gen_YAKE_keyword(path, text_column="text", n_rows=None):
    data_df = pd.read_parquet(path)

    if n_rows:
        data_df = data_df.head(n_rows)
    
    # Apply multiprocessing
    data_df['yake'] = apply_multiprocessing_with_progress(data_df, add_yake_keyword_column, os.cpu_count() // 2, text_column)
    data_df["yake"] = data_df["yake"].apply(json.dumps)

    output_file_name = path.replace(".parquet", "_yake.parquet")
    data_df.to_parquet(output_file_name, index=False)

    return data_df



In [23]:
n_rows = 100
path = "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count.parquet"
df = gen_YAKE_keyword(path, "text", n_rows)


Extracting Keywords: 100%|██████████| 100/100 [00:09<00:00, 11.04it/s]


In [4]:
data_df = pd.read_parquet("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/data/parsed_sde_init_check_with_title_filtered_word_count.parquet")

In [ ]:
df[["text", "yake"]].head(5)

np.int64(2596262)

In [ ]:
# 2596262

import pandas as pd

# Sample DataFrame with a long text
data = {'id': [1], 'text': ['abcdefghijklmnopqrstuvwxyz']}  # 26 characters
df = pd.DataFrame(data)

# Function to split long text into multiple rows
def split_long_text(df, text_column, max_length=1000000):
    new_rows = []
    
    for index, row in df.iterrows():
        text = row[text_column]
        
        # Split text into chunks of max_length
        for i in range(0, len(text), max_length):
            new_row = row.copy()  # Copy other columns
            new_row[text_column] = text[i:i+max_length]  # Assign chunk
            new_rows.append(new_row)
    
    return pd.DataFrame(new_rows)

# Apply function with max_length=10
df_split = split_long_text(df, 'text', max_length=10)

# Check the result
print(df_split)


   id        text
0   1  abcdefghij
0   1  klmnopqrst
0   1      uvwxyz
